# Assignment 31: Conversational PDF Q&A Chatbot with Message History

**Student:** Abhishek Thakare

This is basically Assignment 28 again but pointed at an actual PDF instead of
plain text files - same conversational RAG pattern (retriever + message
history + trimming), just with `PyPDFLoader` doing the loading this time.

I put together an actual PDF for this - `Employee_Handbook.pdf` - which is
the same onboarding/leave/reimbursement/IT-support/code-of-conduct content
from my earlier assignments, just properly formatted as one 6-page document
instead of scattered across separate `.txt` files. Long enough to genuinely
need chunking rather than fitting in one block.

The core logic (loading, splitting, the retriever, the conversational chain,
and history trimming) all lives in `pdf_chat.py`, imported here so this
notebook is testing the same code a real app would use, not a separate copy
of it.

**Same honest note as Assignment 25/28:** no working OpenAI credits, so this
is **Ollama running `llama3.2`** locally again, since the restriction here is
"LangChain + any LLM".


## Before running this

- Ollama running locally with `llama3.2` pulled.
- `data/Employee_Handbook.pdf` in a `data/` folder next to this notebook.
- Same as always - if Ollama or the embedding model isn't reachable, the
  retriever/chain cells print a clear message and set things to `None`
  instead of crashing.


In [1]:
# Run this only if something is missing in your environment
# %pip install -U langchain-core langchain-community langchain-text-splitters langchain-huggingface langchain-ollama faiss-cpu sentence-transformers pypdf

## PART 1 — PDF Ingestion & Preprocessing

### Task 1: Load PDF Documents

Using `PyPDFLoader` this time instead of `TextLoader` - everything else about
loading documents into LangChain stays the same.


In [2]:
from pdf_chat import load_pdfs

pdf_docs = load_pdfs(["data/Employee_Handbook.pdf"])

print("Number of pages loaded:", len(pdf_docs))
print("\nSample content from the first page:\n")
print(pdf_docs[0].page_content[:400])


Number of pages loaded: 6

Sample content from the first page:

Personal Knowledge Assistant - Employee Handbook
This handbook covers everything a new hire needs for their first few weeks - onboarding,
leave, reimbursements, IT support, and the code of conduct training requirement. It's the
same reference material the Personal Knowledge Assistant project has been built around
across these assignments, just put together as one proper document this time instead 


### Task 2: Text Splitting

Same splitter as every other RAG assignment I've done - `RecursiveCharacterTextSplitter`
with an explicit `chunk_size` and `chunk_overlap`.


In [3]:
from pdf_chat import split_documents

pdf_chunks = split_documents(pdf_docs, chunk_size=500, chunk_overlap=100)

print("chunk_size=500, chunk_overlap=100")
print("PDF split into", len(pdf_chunks), "chunks")


chunk_size=500, chunk_overlap=100
PDF split into 15 chunks


## PART 2 — Embeddings & Vector Store

### Task 3 & 4: Create Embeddings + Vector Store Setup

Hugging Face embeddings again - local, no API key needed - stored in FAISS,
same combo I've used since Assignment 25.


In [4]:
embeddings = None
retriever = None

try:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    from pdf_chat import build_retriever
    retriever = build_retriever(["data/Employee_Handbook.pdf"], embeddings, k=3)

    print("Vector store built. Quick test search:")
    for doc in retriever.invoke("What is the leave policy?"):
        print("-", doc.page_content[:100].replace("\n", " "))
except Exception as e:
    print("Couldn't build the vector store:", e)
    print("(Needs internet access the first time, to download the embedding model.)")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store built. Quick test search:
- 2. Leave Policy Full-time employees accrue 18 paid leaves per calendar year. Unlike some companies t
- the leave policy, the reimbursement process, and the mandatory code of conduct training. New hires a
- equivalent for forfeited leave. Requesting Leave Planned leave requests go through the HR portal and


## PART 3 — Conversational Prompt with Message History

### Task 5: RAG Prompt Template

Defined in `pdf_chat.py` as `pdf_qa_prompt` - system message with the
grounding instructions (answer only from the PDF context, say "I don't know"
otherwise), a `MessagesPlaceholder` for chat history, and the current
question as a human message. Same shape as Assignment 28, just with "PDF"
in the wording instead of "documents" generically.


In [5]:
from pdf_chat import pdf_qa_prompt

rendered = pdf_qa_prompt.format_messages(
    context="(sample retrieved PDF context would go here)",
    chat_history=[],
    question="What is the leave policy?",
)
for msg in rendered:
    print(f"[{msg.type}] {msg.content[:200]}")


[system] You are a Conversational PDF Assistant. Answer the question using ONLY the PDF context below - do not use outside knowledge. If the answer isn't in the context, say 'I don't know based on the PDF.' in
[human] What is the leave policy?


## PART 4 — Conversational RAG Chain

### Task 6: Build Conversational RAG Chain

`User Question -> Retriever -> PDF Context -> Prompt + Message History -> LLM -> Answer`,
same LCEL shape as the text-based version from Assignment 28.


In [6]:
llm = None
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model="llama3.2", temperature=0.2)
    llm.invoke("say ok")
    print("Ollama is up, llama3.2 responded.")
except Exception as e:
    llm = None
    print("Couldn't reach Ollama:", e)
    print("The chain cells below will print a placeholder instead of a real answer.")


Couldn't reach Ollama: [WinError 10061] No connection could be made because the target machine actively refused it
The chain cells below will print a placeholder instead of a real answer.


In [7]:
from pdf_chat import build_conversational_chain

pdf_chain = None
if retriever is not None and llm is not None:
    pdf_chain = build_conversational_chain(retriever, llm)
    print("Conversational RAG chain is ready.")
else:
    print("Skipping - need both a retriever and Ollama available to build the real chain.")


Skipping - need both a retriever and Ollama available to build the real chain.


### Task 7 & 8: Maintain + Trim Message History

Pulled this into a small `ConversationalPDFChatbot` class in `pdf_chat.py` -
it keeps its own `chat_history`, appends both sides of every turn, and trims
down to the most recent N messages afterward, so I'm not managing that list
by hand in every cell. The trimming itself doesn't need the LLM at all, so I
can verify it directly with a fake conversation.


In [8]:
from pdf_chat import ConversationalPDFChatbot, trim_history
from langchain_core.messages import HumanMessage, AIMessage

# quick, LLM-free check that trimming actually works before trusting it
# inside the chatbot class
fake_history = []
for i in range(1, 5):
    fake_history.append(HumanMessage(content=f"question {i}"))
    fake_history.append(AIMessage(content=f"answer {i}"))

print("Before trimming:", len(fake_history), "messages")
print("After trimming to max_messages=4:", len(trim_history(fake_history, max_messages=4)), "messages")


Before trimming: 8 messages
After trimming to max_messages=4: 4 messages


In [9]:
chatbot = None
if pdf_chain is not None:
    chatbot = ConversationalPDFChatbot(pdf_chain, max_history_messages=6)
    print("Chatbot ready.")
else:
    print("No working chain - chatbot will just print placeholders below.")

def ask(question):
    if chatbot is None:
        return "[No working chain right now - Ollama or the vector store isn't available]"
    try:
        return chatbot.ask(question)
    except Exception as e:
        return f"[Question failed: {e}]"

print(ask("What is the leave policy?"))


No working chain - chatbot will just print placeholders below.
[No working chain right now - Ollama or the vector store isn't available]


## PART 5 — Multi-Turn Conversation Testing

### Task 9: Follow-Up Q&A Testing

Resetting history first, then an initial factual question straight from the
PDF, a follow-up that only makes sense with that answer in mind, and a short
clarification question.


In [10]:
if chatbot is not None:
    chatbot.reset()

turns = [
    "What is the leave policy?",                      # initial factual question from the PDF
    "What about carrying leaves over to next year?",     # follow-up referencing the previous answer
    "Can you explain that more simply?",                  # clarification question
]

for q in turns:
    print("-" * 60)
    print("You:", q)
    print("Bot:", ask(q))

print("\nMessages in history after this test:", len(chatbot.chat_history) if chatbot else 0)


------------------------------------------------------------
You: What is the leave policy?
Bot: [No working chain right now - Ollama or the vector store isn't available]
------------------------------------------------------------
You: What about carrying leaves over to next year?
Bot: [No working chain right now - Ollama or the vector store isn't available]
------------------------------------------------------------
You: Can you explain that more simply?
Bot: [No working chain right now - Ollama or the vector store isn't available]

Messages in history after this test: 0


Same limitation as my last couple of notebooks - couldn't actually confirm
grounded, context-preserving answers here since neither Ollama nor the
embedding model came through in this environment. If this were working, I'd
want the second answer to clearly build on the carry-forward detail from the
first rather than re-explaining the whole leave policy, and the third to
simplify the *same* information instead of drifting to something new.


## PART 6 — Mini Project: Conversational PDF Chatbot

### Task 10: Build Final Chatbot Application

Everything from Parts 1-5 is already wrapped into `ConversationalPDFChatbot`
in `pdf_chat.py`, so the "final app" here is really just this small loop on
top of it. A Streamlit UI was optional - kept it to a plain input loop since
the restriction was to focus on conversational grounding and memory, not the
interface.


In [11]:
def run_pdf_chatbot():
    if chatbot is not None:
        chatbot.reset()
    print("Conversational PDF Assistant - type 'exit' to quit\n")

    while True:
        user_input = input("You: ")
        if user_input.strip().lower() == "exit":
            print("Bot: Catch you later!")
            break

        print("Bot:", ask(user_input))
        print()

# not calling this automatically since it blocks on input() -
# uncomment to actually chat with it interactively
# run_pdf_chatbot()


## Task 11: Observations & Insights

**1. Difference between plain PDF Q&A and conversational PDF Q&A**
Plain PDF Q&A retrieves and answers each question in isolation - it has no
idea what was asked a moment ago. Conversational PDF Q&A adds `chat_history`
into the prompt via `MessagesPlaceholder`, so a follow-up that only makes
sense in light of the previous answer actually has something to resolve
against. The retrieval side is identical either way; it's purely a prompt
change.

**2. Role of message history in follow-up questions**
Without it, something like "what about carrying it over?" is ambiguous on
its own - carrying *what* over? The history is what supplies the missing
subject, letting the model connect a vague follow-up back to a specific
detail from an earlier answer instead of failing to understand the question
at all.

**3. Trade-offs between long memory and performance**
Keeping the full conversation forever means every call sends more text to
the model, which gets slower and eventually won't fit in the context window.
Trimming keeps calls fast and bounded, at the cost of genuinely forgetting
anything before the cutoff - if a much earlier point in a long conversation
gets referenced again, a trimmed history simply won't have it anymore.

**4. How trimming affects answer quality**
For anything recent, trimming has essentially no downside - the model still
has the retrieved PDF context plus enough recent conversation to follow
along fine. The risk is purely for long-range callbacks: if someone circles
back to something from much earlier than the trim window covers, the bot
will either say it doesn't know or misread the follow-up, since that part of
the conversation is genuinely gone from what it sees.


## Final note

Nothing about the retrieval pipeline changed from Assignment 28 - same
load/split/embed/store/retrieve pattern, just against `PyPDFLoader` output
instead of plain text files. The conversational piece (`MessagesPlaceholder`,
trimming) is also identical in shape - wrapping it into a small
`ConversationalPDFChatbot` class this time mostly just made it easier to
reuse the exact same object across the trimming test, the multi-turn test,
and the final chatbot loop, instead of managing `chat_history` by hand in
each section separately.
